# Inverse Problem for the Steady-State 2D RTE using Vanilla-PINNs

In this notebook, we solve the **inverse** 2D Radiative Transfer Equation (RTE) in a participating square medium using a **VANILLA-PINN** approach. The forward configuration is Case 2 (Crosbie & Schrenker); here the scattering coefficient $\sigma$ is **unknown** and is recovered from noisy measurements of the incident radiation $G$.

### Case 2 — Inverse (2D)
* **Geometry:** $0 \le x \le L_x$ and $0 \le y \le L_y$ with $L_x = L_y = 1.0\text{ m}$ (Square medium).
* **Equation (2D RTE):**
  $$\mu \frac{\partial I}{\partial x} + \eta \frac{\partial I}{\partial y} + (\kappa + \sigma) I(x, y, \mu, \eta) = \frac{\sigma}{4\pi} G(x, y)$$
* **Unknown:** the scattering coefficient $\sigma$ (true value $\sigma^\star = 1.0\text{ m}^{-1}$), with $\kappa = 0$ fixed.
* **Data:** noisy measurements $G^{obs}$ of the incident radiation at scattered interior sensor points, generated by the DOM reference.
* **Boundary Conditions (BC) on incoming directions:**
  * Top boundary ($y = 1$, for $\eta < 0$): $I = 1.0$ (Diffuse radiation).
  * The three other walls: $I = 0.0$.

The network approximates $I(x, y, \mu, \eta)$ and $\sigma$ is a **learnable parameter**. Three losses are minimized jointly: the boundary conditions, the PDE residual (which depends on $\sigma$), and a **data loss** matching $G$ at the sensors. Recovering $\sigma \to \sigma^\star$ validates the inverse solver.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

## 1. PINN Architecture Definition
Definition of the standard Neural Network (Multi-Layer Perceptron) to approximate the intensity $I(x, y, \mu, \eta)$.

In [ ]:
class PinnRFEEq2D(nn.Module):
    def __init__(self, hidden_dim=64, num_layers=4):
        super().__init__()
        layers = []
        layers.append(nn.Linear(4, hidden_dim))
        layers.append(nn.Tanh())
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.Tanh())
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

## 2. Double Quadrature Setup for the Unit Sphere
Setting up a tensor product of Gauss-Legendre quadratures for $\xi = \cos\theta$ and a uniform midpoint rule in $\phi$ to perform solid angle integration.

In [ ]:
def init_quadrature(N_theta=8, N_phi=16, device='cpu'):
    nodes_xi, weights_xi = np.polynomial.legendre.leggauss(N_theta)
    nodes_xi = 0.5 * (nodes_xi + 1.0)

    nodes_phi = (np.arange(N_phi) + 0.5) * 2.0 * np.pi / N_phi
    weights_phi = np.full(N_phi, 2.0 * np.pi / N_phi)

    quad_mu_list = []
    quad_eta_list = []
    quad_w_list = []

    for i in range(N_theta):
        for j in range(N_phi):
            xi = nodes_xi[i]
            phi = nodes_phi[j]
            w = weights_xi[i] * weights_phi[j]

            mu = np.sqrt(1.0 - xi**2) * np.cos(phi)
            eta = np.sqrt(1.0 - xi**2) * np.sin(phi)

            quad_mu_list.append(mu)
            quad_eta_list.append(eta)
            quad_w_list.append(w)

    quad_mu = torch.tensor(quad_mu_list, dtype=torch.float32).view(-1, 1).to(device)
    quad_eta = torch.tensor(quad_eta_list, dtype=torch.float32).view(-1, 1).to(device)
    quad_w = torch.tensor(quad_w_list, dtype=torch.float32).view(-1, 1).to(device)
    return quad_mu, quad_eta, quad_w

## 3. Sampling and Collocation Points Generation
Functions to generate collocation points in the 4D domain $(x, y, \mu, \eta)$ and boundary points on the four limits for incoming directions.

In [ ]:
def generer_points_collocation(n_pde):
    x = torch.rand(n_pde, 1)
    y = torch.rand(n_pde, 1)
    xi = torch.rand(n_pde, 1) * 2.0 - 1.0
    phi = torch.rand(n_pde, 1) * 2.0 * np.pi
    mu = torch.sqrt(1.0 - xi**2) * torch.cos(phi)
    eta = torch.sqrt(1.0 - xi**2) * torch.sin(phi)
    return x.float(), y.float(), mu.float(), eta.float()

def generer_points_bords(n_bords):
    n_edge = n_bords // 4

    # Left edge: x = 0, y in [0,1], mu > 0
    x_left = torch.zeros(n_edge, 1)
    y_left = torch.rand(n_edge, 1)
    xi_left = torch.rand(n_edge, 1) * 2.0 - 1.0
    phi_left = (torch.rand(n_edge, 1) - 0.5) * np.pi
    mu_left = torch.sqrt(1.0 - xi_left**2) * torch.cos(phi_left)
    eta_left = torch.sqrt(1.0 - xi_left**2) * torch.sin(phi_left)

    # Right edge: x = 1, y in [0,1], mu < 0
    x_right = torch.ones(n_edge, 1)
    y_right = torch.rand(n_edge, 1)
    xi_right = torch.rand(n_edge, 1) * 2.0 - 1.0
    phi_right = torch.rand(n_edge, 1) * np.pi + 0.5 * np.pi
    mu_right = torch.sqrt(1.0 - xi_right**2) * torch.cos(phi_right)
    eta_right = torch.sqrt(1.0 - xi_right**2) * torch.sin(phi_right)

    # Bottom edge: x in [0,1], y = 0, eta > 0
    x_bottom = torch.rand(n_edge, 1)
    y_bottom = torch.zeros(n_edge, 1)
    xi_bottom = torch.rand(n_edge, 1) * 2.0 - 1.0
    phi_bottom = torch.rand(n_edge, 1) * np.pi
    mu_bottom = torch.sqrt(1.0 - xi_bottom**2) * torch.cos(phi_bottom)
    eta_bottom = torch.sqrt(1.0 - xi_bottom**2) * torch.sin(phi_bottom)

    # Top edge: x in [0,1], y = 1, eta < 0
    x_top = torch.rand(n_edge, 1)
    y_top = torch.ones(n_edge, 1)
    xi_top = torch.rand(n_edge, 1) * 2.0 - 1.0
    phi_top = torch.rand(n_edge, 1) * np.pi + np.pi
    mu_top = torch.sqrt(1.0 - xi_top**2) * torch.cos(phi_top)
    eta_top = torch.sqrt(1.0 - xi_top**2) * torch.sin(phi_top)

    return (
        x_left.float(), y_left.float(), mu_left.float(), eta_left.float(),
        x_right.float(), y_right.float(), mu_right.float(), eta_right.float(),
        x_bottom.float(), y_bottom.float(), mu_bottom.float(), eta_bottom.float(),
        x_top.float(), y_top.float(), mu_top.float(), eta_top.float(),
    )

## 4. Loss Functions Definition
Three losses: boundary conditions (BC), the PDE residual (CLP) — which now depends on the learnable $\sigma$ — and the **data loss** matching $G$ at the sensor points. `calc_G` computes $G$ (with gradient) at arbitrary points.

In [ ]:
def calc_bc_loss(model,
                 x_l, y_l, mu_l, eta_l,
                 x_r, y_r, mu_r, eta_r,
                 x_b, y_b, mu_b, eta_b,
                 x_t, y_t, mu_t, eta_t):

    pred_l = model(torch.cat([x_l, y_l, mu_l, eta_l], dim=1))
    pred_r = model(torch.cat([x_r, y_r, mu_r, eta_r], dim=1))
    pred_b = model(torch.cat([x_b, y_b, mu_b, eta_b], dim=1))
    pred_t = model(torch.cat([x_t, y_t, mu_t, eta_t], dim=1))

    loss_l = torch.mean((pred_l - 0.0) ** 2)
    loss_r = torch.mean((pred_r - 0.0) ** 2)
    loss_b = torch.mean((pred_b - 0.0) ** 2)
    loss_t = torch.mean((pred_t - 1.0) ** 2)

    return loss_l + loss_r + loss_b + 5.0 * loss_t

def calc_G(model, x, y, quad_mu, quad_eta, quad_w):
    N = x.shape[0]
    N_q = quad_mu.shape[0]
    x_expanded = x.repeat(1, N_q)
    y_expanded = y.repeat(1, N_q)
    mu_expanded = quad_mu.t().repeat(N, 1)
    eta_expanded = quad_eta.t().repeat(N, 1)
    inputs_quad = torch.stack([x_expanded, y_expanded, mu_expanded, eta_expanded], dim=2).view(-1, 4)
    I_quad = model(inputs_quad).view(N, N_q)
    return torch.sum(I_quad * quad_w.t(), dim=1, keepdim=True)

def calc_clp_loss(model, x, y, mu, eta, quad_mu, quad_eta, quad_w, kappa, sigma):
    x.requires_grad_(True)
    y.requires_grad_(True)

    I_pred = model(torch.cat([x, y, mu, eta], dim=1))

    I_x = torch.autograd.grad(I_pred, x, torch.ones_like(I_pred), create_graph=True)[0]
    I_y = torch.autograd.grad(I_pred, y, torch.ones_like(I_pred), create_graph=True)[0]

    G = calc_G(model, x, y, quad_mu, quad_eta, quad_w)

    residual = mu * I_x + eta * I_y + (kappa + sigma) * I_pred - (sigma / (4.0 * np.pi)) * G
    return torch.mean(residual ** 2)

def calc_data_loss(model, x_obs, y_obs, G_obs, quad_mu, quad_eta, quad_w):
    G_pred = calc_G(model, x_obs, y_obs, quad_mu, quad_eta, quad_w)
    return torch.mean((G_pred - G_obs) ** 2)

## 5. Reference DOM and Synthetic Measurements
The DOM solves the forward problem at the **true** $\sigma^\star = 1.0$ and provides both the validation reference and the noisy sensor data $G^{obs}$ used to drive the inversion.

In [ ]:
# Solveur DOM 2D de reference (identique au cas 2), utilise pour generer les donnees.

def resoudre_dom(M=201, N_xi=12, N_phi=48, beta=1.0, sigma_dom=1.0, tol=1e-8, max_iter=2000):
    dx = 1.0 / (M - 1)

    nx, wx = np.polynomial.legendre.leggauss(N_xi)
    xi = 0.5 * (nx + 1.0)
    phi = (np.arange(N_phi) + 0.5) * 2.0 * np.pi / N_phi
    XI, PHI = np.meshgrid(xi, phi, indexing='ij')
    W = np.outer(wx, np.full(N_phi, 2.0 * np.pi / N_phi)).ravel()
    st = np.sqrt(1.0 - XI**2)
    MU = (st * np.cos(PHI)).ravel()
    ETA = (st * np.sin(PHI)).ravel()

    quadrants = []
    for smu in (1, -1):
        for seta in (1, -1):
            sel = (np.sign(MU) == smu) & (np.sign(ETA) == seta)
            quadrants.append((smu, seta, MU[sel], ETA[sel], W[sel]))

    def balayage(smu, seta, mu, eta, S):
        a = np.abs(mu)[:, None] / dx
        b = np.abs(eta)[:, None] / dx
        I = np.zeros((mu.size, M, M))
        bc_y = 1.0 if seta < 0 else 0.0   # paroi haute (y=1) chaude, les autres a 0
        Sf = S[::smu, ::seta]
        If = I[:, ::smu, ::seta]
        denom = a + b + beta
        for d in range(2 * M - 1):
            ii = np.arange(max(0, d - M + 1), min(d, M - 1) + 1)
            jj = d - ii
            Ix = If[:, np.where(ii > 0, ii - 1, 0), jj]
            Ix[:, ii == 0] = 0.0
            Iy = If[:, ii, np.where(jj > 0, jj - 1, 0)]
            Iy[:, jj == 0] = bc_y
            If[:, ii, jj] = (a * Ix + b * Iy + Sf[ii, jj][None, :]) / denom
        return I

    G = np.zeros((M, M))
    for it in range(max_iter):
        S = (sigma_dom / (4.0 * np.pi)) * G
        Gn = np.zeros_like(G)
        for smu, seta, mu, eta, w in quadrants:
            Gn += np.tensordot(w, balayage(smu, seta, mu, eta, S), axes=(0, 0))
        diff = np.max(np.abs(Gn - G))
        G = Gn
        if diff < tol:
            break
    print(f"DOM converge en {it} iterations (diff = {diff:.2e})")
    return G

sigma_true = 1.0
kappa = 0.0

M_dom = 201
G_dom = resoudre_dom(M=M_dom, beta=kappa + sigma_true, sigma_dom=sigma_true)
grid_dom = np.linspace(0.0, 1.0, M_dom)
mid_dom = (M_dom - 1) // 2
print(f"G(0.5, 0.5) DOM = {G_dom[mid_dom, mid_dom]:.4f}   (exact = pi = {np.pi:.4f})")

# Generation des mesures bruitees : capteurs interieurs aleatoires + 2% de bruit
np.random.seed(0)
n_obs = 200
xy_obs = np.random.rand(n_obs, 2) * 0.8 + 0.1
ix = np.round(xy_obs[:, 0] * (M_dom - 1)).astype(int)
iy = np.round(xy_obs[:, 1] * (M_dom - 1)).astype(int)
G_clean = G_dom[ix, iy]
G_noisy = G_clean * (1.0 + 0.02 * np.random.randn(n_obs))
print(f"{n_obs} capteurs, bruit relatif 2%")

## 6. Hardware (Device), Model, Learnable $\sigma$ and Optimizer Initialization
The scattering coefficient is a learnable `nn.Parameter` initialized far from the truth ($\sigma_0 = 0.5$); the optimizer updates the network **and** $\sigma$ jointly.

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

quad_mu, quad_eta, quad_w = init_quadrature(N_theta=16, N_phi=32, device=device)

n_pde = 4000
n_bords = 4000

x_colloc, y_colloc, mu_colloc, eta_colloc = generer_points_collocation(n_pde)
x_colloc = x_colloc.to(device); y_colloc = y_colloc.to(device)
mu_colloc = mu_colloc.to(device); eta_colloc = eta_colloc.to(device)

(
    x_l, y_l, mu_l, eta_l,
    x_r, y_r, mu_r, eta_r,
    x_b, y_b, mu_b, eta_b,
    x_t, y_t, mu_t, eta_t
) = generer_points_bords(n_bords)
x_l, y_l, mu_l, eta_l = x_l.to(device), y_l.to(device), mu_l.to(device), eta_l.to(device)
x_r, y_r, mu_r, eta_r = x_r.to(device), y_r.to(device), mu_r.to(device), eta_r.to(device)
x_b, y_b, mu_b, eta_b = x_b.to(device), y_b.to(device), mu_b.to(device), eta_b.to(device)
x_t, y_t, mu_t, eta_t = x_t.to(device), y_t.to(device), mu_t.to(device), eta_t.to(device)

x_obs = torch.tensor(xy_obs[:, 0:1], dtype=torch.float32).to(device)
y_obs = torch.tensor(xy_obs[:, 1:2], dtype=torch.float32).to(device)
G_obs = torch.tensor(G_noisy, dtype=torch.float32).view(-1, 1).to(device)

modele = PinnRFEEq2D(hidden_dim=64, num_layers=4).to(device)
sigma_param = torch.nn.Parameter(torch.tensor(0.5, device=device))   # inconnue, initialisee loin de la verite
print(f"sigma initial = {sigma_param.item():.4f}  (verite = {sigma_true})")

## 7. Model Training (Adam)
Joint minimization of BC + PDE residual + data loss over the network weights and $\sigma$.

In [ ]:
w_data = 100.0
optimizer = optim.Adam([
    {"params": modele.parameters(), "lr": 2e-3},
    {"params": [sigma_param], "lr": 5e-3},   # sigma a son propre pas d'apprentissage
])
epochs = 6000
hist_sigma = []

for epoch in range(epochs):
    optimizer.zero_grad()
    loss_bc = calc_bc_loss(modele,
                           x_l, y_l, mu_l, eta_l,
                           x_r, y_r, mu_r, eta_r,
                           x_b, y_b, mu_b, eta_b,
                           x_t, y_t, mu_t, eta_t)
    loss_clp = calc_clp_loss(modele, x_colloc, y_colloc, mu_colloc, eta_colloc,
                             quad_mu, quad_eta, quad_w, kappa, sigma_param)
    loss_data = calc_data_loss(modele, x_obs, y_obs, G_obs, quad_mu, quad_eta, quad_w)
    loss_totale = loss_bc + loss_clp + w_data * loss_data
    loss_totale.backward()
    optimizer.step()

    hist_sigma.append(sigma_param.item())
    if epoch % 200 == 0:
        print(f"Epoque {epoch:04d} | Loss {loss_totale.item():.2e} | "
              f"CLP {loss_clp.item():.2e} | BC {loss_bc.item():.2e} | "
              f"Data {loss_data.item():.2e} | sigma {sigma_param.item():.4f}")

## 8. Model Training (L-BFGS)
Fine-tuning of the **network only** with L-BFGS, with $\sigma$ **frozen** at its Adam-identified value. Reason: mixing a single physical scalar with the ~13k network weights inside one L-BFGS makes $\sigma$ diverge (an ill-posed local minimum where the network overfits the noisy data at a wrong $\sigma$). The parameter is identified by Adam; L-BFGS only polishes the field.

In [ ]:
sigma_param.requires_grad_(False)   # sigma gele : identifie par Adam

def closure():
    optimizer_lbfgs.zero_grad()
    loss_bc = calc_bc_loss(modele,
                           x_l, y_l, mu_l, eta_l,
                           x_r, y_r, mu_r, eta_r,
                           x_b, y_b, mu_b, eta_b,
                           x_t, y_t, mu_t, eta_t)
    loss_clp = calc_clp_loss(modele, x_colloc, y_colloc, mu_colloc, eta_colloc,
                             quad_mu, quad_eta, quad_w, kappa, sigma_param)
    loss_data = calc_data_loss(modele, x_obs, y_obs, G_obs, quad_mu, quad_eta, quad_w)
    loss_totale = loss_bc + loss_clp + w_data * loss_data
    loss_totale.backward()
    return loss_totale

optimizer_lbfgs = optim.LBFGS(modele.parameters(),
                              line_search_fn="strong_wolfe", max_iter=20)
lbfgs_epochs = 300

for epoch in range(lbfgs_epochs):
    loss = optimizer_lbfgs.step(closure)
    hist_sigma.append(sigma_param.item())
    if epoch % 20 == 0:
        print(f"Epoque LBFGS {epoch:03d} | Loss {loss.item():.2e} | sigma {sigma_param.item():.4f}")

print(f"\nsigma recupere = {sigma_param.item():.4f}   (verite = {sigma_true})")
print(f"erreur relative sur sigma = {abs(sigma_param.item() - sigma_true) / sigma_true * 100:.2f} %")

## 9. Results: Recovered $\sigma$ and Field Comparison vs DOM

In [ ]:
# Convergence de sigma
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].plot(hist_sigma, 'b-', lw=1.5, label="sigma (PINN)")
ax[0].axhline(sigma_true, color='k', ls='--', label=f"verite = {sigma_true}")
ax[0].set_xlabel("iteration"); ax[0].set_ylabel(r"$\sigma$")
ax[0].set_title(f"Convergence de sigma (final = {sigma_param.item():.4f})")
ax[0].grid(True); ax[0].legend()

# G(0.5, y) : PINN (sigma recupere) vs DOM (verite)
def evaluer_G_np(model, x_grid, y_grid, quad_mu, quad_eta, quad_w, device):
    X, Y = np.meshgrid(x_grid, y_grid)
    xt = torch.tensor(X.ravel(), dtype=torch.float32).view(-1, 1).to(device)
    yt = torch.tensor(Y.ravel(), dtype=torch.float32).view(-1, 1).to(device)
    with torch.no_grad():
        G = calc_G(model, xt, yt, quad_mu, quad_eta, quad_w)
    return G.cpu().numpy().reshape(Y.shape)

y_line = np.linspace(0.0, 1.0, 100)
G_x05 = evaluer_G_np(modele, np.array([0.5]), y_line, quad_mu, quad_eta, quad_w, device).flatten()
ax[1].plot(y_line, G_x05, 'r-', lw=2, label="PINN")
ax[1].plot(grid_dom, G_dom[mid_dom, :], 'k--', lw=1.5, label="DOM (verite)")
ax[1].plot(0.5, np.pi, 'b*', ms=12, label=r"exact $G(0.5,0.5)=\pi$")
ax[1].set_xlabel("Position $y$"); ax[1].set_ylabel("$G(0.5, y)$")
ax[1].set_title("Incident Radiation $G(0.5, y)$"); ax[1].grid(True); ax[1].legend()

plt.tight_layout()
fig.savefig("inverse_cas2_sigma.png", dpi=150)
plt.show()

print(f"sigma recupere = {sigma_param.item():.4f}   (verite = {sigma_true})")
print(f"erreur relative sur sigma = {abs(sigma_param.item() - sigma_true) / sigma_true * 100:.2f} %")